# N14. Deterministic retrain, arm F, clean condition, and why black fails

What N12 and N13 established on development data:

1. With heavy occlusion 40 to 47% of markers tie under hard decoding, and lowest id tie
   breaking costs up to 8 points.
2. The published decoder guesses hidden cells at random with near full confidence, for
   every occluder colour. Grey and white occluders also damage visible cells (visible bit
   accuracy 79 to 90%) while black barely does (92 to 95%). That is why soft decoding loses
   to fair hard decoding under grey and white and wins under black.
3. Retrained decoders need soft decoding (up to 43 points over hard) and on grey and white
   they are already close to their own oracle.
4. On black every model falls about 35 points short of its oracle, so the black failure is
   about not detecting which cells are hidden.
5. Masked loss plus erasure output (arm C) reaches its oracle on grey and white and fails on
   black, which it never saw in training.

Why this notebook exists:

* **Checkpoint provenance.** In N13, arm B seeds 0 to 2 came from older code, and training
  was not deterministic, so the same seed gave different weights. Everything is retrained
  here into a fresh folder with deterministic settings. Arms B and M have identical
  settings, so their weights must now match exactly, which is checked.
* **Partial checkpoints.** N13 saved the checkpoint during training, so an interrupted run
  would later be loaded as if finished. Here a checkpoint is written only when the run
  completes, and it carries a signature of the settings that made it.
* **Arm F** is arm C (masked loss, erasure output) trained with random brightness
  occluders. If its erasure output learns to flag black occluders, the black gap closes and
  finding 4 becomes a mechanism.
* **Clean condition.** No occluder, to measure domain shift between the published decoder
  and the retrained arms.
* **Confidence cap.** Soft decoding with probabilities clipped to [eps, 1 minus eps], so one
  confidently wrong bit cannot dominate. eps is chosen on videos 1 and 2 and tested on 3 to 5.
* **Where visible damage happens.** Visible cell accuracy split by distance from the
  occluder.
* **Calibration of retrained arms**, including how often the erasure output flags hidden
  and visible cells.

Statistics use Welch's unpaired test across seeds. A difference is called equivalent to zero
only when the whole interval lies inside plus or minus 1 point. Everything runs on
development data; video 6 stays untouched.

**Before running: Runtime, Restart session, then Run all.** The deterministic setting must
be in place before torch starts, so a session that already imported torch is not enough.


In [1]:
!pip -q install opencv-contrib-python==4.10.0.84 h5py 2>&1 | tail -1

In [2]:
%%writefile deeparuco_port.py
"""
Port the pretrained DeepArUco++ decoder from Keras .h5 to PyTorch, without TensorFlow.

The published decoder (models/dec_new.h5 in AVAuco/deeparuco) is a Keras 2.11 model:

    Input(32, 32, 1)
    Conv2D(8, 5x5, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(4, 3x3, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(1, 1x1, valid, sigmoid)   ->  (6, 6, 1)

Only h5py is needed to read the weights, so the repo's Python 3.9 + TensorFlow
environment is never required for baseline inference.

IMPORTANT PORTING DETAIL
    Keras PReLU defaults to shared_axes=None, so alpha is PER-ELEMENT, not per-channel:
    the stored alphas have shape (28, 28, 8) and (12, 12, 4). torch.nn.PReLU only
    supports per-channel alpha, so using it here silently produces a different model.
    SpatialPReLU below keeps the full alpha tensor.

Layout conversions:
    Keras conv kernel (kh, kw, in, out) -> torch (out, in, kh, kw)
    Keras PReLU alpha (h, w, c)         -> torch (c, h, w)
    Keras 'valid' padding               -> torch padding=0

Usage:
    from deeparuco_port import load_weights, NumpyDecoder, TorchDecoder, verify_port
    verify_port('models/dec_new.h5')          # run this first, once
    dec = TorchDecoder.from_h5('models/dec_new.h5')
    probs = dec(x)                            # x: (N, 1, 32, 32) float in [0, 1]

Upstream code is AGPL-3.0. This module reads a weights file; it does not copy source.
Cite Berral-Soler et al., Image and Vision Computing 152 (2024) 105313.
"""

import numpy as np

DEFAULT_H5 = "models/dec_new.h5"


# --------------------------------------------------------------------------- weights

def load_weights(h5_path=DEFAULT_H5):
    """Read decoder weights from the Keras .h5. Returns a dict of numpy arrays."""
    import h5py

    store = {}
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset) and name.startswith("model_weights/"):
                store[name] = np.array(obj)
        f.visititems(visit)

    def pick(layer, suffix):
        hits = [k for k in store if f"/{layer}/" in k and k.endswith(suffix)]
        if len(hits) != 1:
            raise KeyError(f"expected one match for {layer}:{suffix}, got {hits}")
        return store[hits[0]]

    w = {
        "c1_w": pick("conv2d", "kernel:0"),   "c1_b": pick("conv2d", "bias:0"),
        "p1_a": pick("p_re_lu", "alpha:0"),
        "c2_w": pick("conv2d_1", "kernel:0"), "c2_b": pick("conv2d_1", "bias:0"),
        "p2_a": pick("p_re_lu_1", "alpha:0"),
        "c3_w": pick("conv2d_2", "kernel:0"), "c3_b": pick("conv2d_2", "bias:0"),
    }
    expect = {"c1_w": (5,5,1,8), "c1_b": (8,), "p1_a": (28,28,8),
              "c2_w": (3,3,8,4), "c2_b": (4,), "p2_a": (12,12,4),
              "c3_w": (1,1,4,1), "c3_b": (1,)}
    for k, shape in expect.items():
        assert w[k].shape == shape, f"{k}: got {w[k].shape}, expected {shape}"
    return w


# ------------------------------------------------------------------- numpy reference

class NumpyDecoder:
    """Reference implementation. Verified against synthetic markers: 4/5 exact."""

    def __init__(self, weights):
        self.w = weights

    @classmethod
    def from_h5(cls, path=DEFAULT_H5):
        return cls(load_weights(path))

    @staticmethod
    def _conv(x, w, b):
        kh, kw, _, co = w.shape
        H, W, _ = x.shape
        out = np.zeros((H - kh + 1, W - kw + 1, co), np.float32)
        for i in range(out.shape[0]):
            for j in range(out.shape[1]):
                out[i, j] = np.tensordot(
                    x[i:i+kh, j:j+kw, :], w, axes=([0, 1, 2], [0, 1, 2])) + b
        return out

    @staticmethod
    def _pool(x):
        h, w, c = x.shape
        return x.reshape(h // 2, 2, w // 2, 2, c).max(axis=(1, 3))

    def __call__(self, marker32):
        """marker32: (32, 32) float in [0, 1]. Returns (6, 6) bit probabilities."""
        w = self.w
        x = np.asarray(marker32, np.float32)[..., None]
        x = self._conv(x, w["c1_w"], w["c1_b"])
        x = np.where(x > 0, x, w["p1_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c2_w"], w["c2_b"])
        x = np.where(x > 0, x, w["p2_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c3_w"], w["c3_b"])
        return (1.0 / (1.0 + np.exp(-np.clip(x[..., 0], -60, 60)))).astype(np.float32)


# -------------------------------------------------------------------------- pytorch

def _torch():
    import torch
    import torch.nn as nn
    return torch, nn


class SpatialPReLU:
    """Created via _build so torch is imported lazily. See module docstring."""


def TorchDecoder_factory():
    torch, nn = _torch()

    class _SpatialPReLU(nn.Module):
        """Per-element PReLU, matching Keras PReLU(shared_axes=None)."""
        def __init__(self, alpha_chw):
            super().__init__()
            self.alpha = nn.Parameter(torch.tensor(alpha_chw, dtype=torch.float32))

        def forward(self, x):
            return torch.where(x > 0, x, self.alpha.unsqueeze(0) * x)

    class _TorchDecoder(nn.Module):
        def __init__(self, weights=None):
            super().__init__()
            self.c1 = nn.Conv2d(1, 8, 5)
            self.c2 = nn.Conv2d(8, 4, 3)
            self.c3 = nn.Conv2d(4, 1, 1)
            self.pool = nn.MaxPool2d(2)
            a1 = np.zeros((8, 28, 28), np.float32) if weights is None \
                 else np.transpose(weights["p1_a"], (2, 0, 1))
            a2 = np.zeros((4, 12, 12), np.float32) if weights is None \
                 else np.transpose(weights["p2_a"], (2, 0, 1))
            self.p1, self.p2 = _SpatialPReLU(a1), _SpatialPReLU(a2)
            if weights is not None:
                self.load_keras(weights)

        def load_keras(self, w):
            with torch.no_grad():
                for conv, kw, kb in ((self.c1, "c1_w", "c1_b"),
                                     (self.c2, "c2_w", "c2_b"),
                                     (self.c3, "c3_w", "c3_b")):
                    conv.weight.copy_(torch.tensor(np.transpose(w[kw], (3, 2, 0, 1))))
                    conv.bias.copy_(torch.tensor(w[kb]))

        @classmethod
        def from_h5(cls, path=DEFAULT_H5):
            return cls(load_weights(path))

        def forward(self, x):
            """x: (N, 1, 32, 32) in [0, 1]. Returns (N, 6, 6) bit probabilities."""
            x = self.pool(self.p1(self.c1(x)))
            x = self.pool(self.p2(self.c2(x)))
            return torch.sigmoid(self.c3(x)).squeeze(1)

    return _TorchDecoder


def __getattr__(name):
    if name == "TorchDecoder":
        return TorchDecoder_factory()
    raise AttributeError(name)


# --------------------------------------------------------------------------- verify

def verify_port(h5_path=DEFAULT_H5, n=16, tol=1e-4, verbose=True):
    """Assert the PyTorch port matches the numpy reference. Run once before use.

    The numpy path is validated; the torch path is validated against it here rather
    than assumed. Do not skip this.
    """
    torch, _ = _torch()
    w = load_weights(h5_path)
    ref = NumpyDecoder(w)
    net = TorchDecoder_factory()(w).eval()

    rng = np.random.default_rng(0)
    x = rng.random((n, 32, 32)).astype(np.float32)
    with torch.no_grad():
        got = net(torch.tensor(x).unsqueeze(1)).numpy()
    want = np.stack([ref(xi) for xi in x])

    err = float(np.abs(got - want).max())
    if verbose:
        print(f"port check: {n} random inputs, max abs difference {err:.2e} (tol {tol:g})")
    assert err < tol, f"PyTorch port disagrees with numpy reference (max diff {err:.2e})"
    if verbose:
        print("PyTorch port matches the numpy reference.")
    return err


def decode_ids(probs, id_to_bits, n_markers=250):
    """Hard-decision decode: nearest codeword by Hamming distance over 4 rotations.

    probs: (6, 6) bit probabilities. Returns (marker_id, rotation, hamming).
    Replace with soft log-likelihood decoding to use the confidences.
    """
    obs = (np.asarray(probs) > 0.5).astype(np.uint8)
    best = (None, None, 99)
    for mid in range(n_markers):
        ref = np.asarray(id_to_bits(mid), np.uint8).reshape(6, 6)
        for rot in range(4):
            h = int(np.sum(np.rot90(obs, rot) != ref))
            if h < best[2]:
                best = (mid, rot, h)
    return best


if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_H5
    w = load_weights(path)
    print("weights loaded:", {k: v.shape for k, v in w.items()})
    try:
        verify_port(path)
    except ImportError:
        print("torch not installed; numpy reference verified only")


Writing deeparuco_port.py


In [3]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # deterministic cuBLAS, must be set before torch
import sys, json, glob, zlib, subprocess, time, itertools
import numpy as np, cv2, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, WORK = '/content/drive/MyDrive/aruco_p2', '/content/work'
except ImportError:
    BASE = os.environ.get('ARUCO_P2_BASE',
                          os.path.join(os.path.expanduser('~'), 'aruco_p2')).replace('\\', '/')
    WORK = f'{BASE}/work'
ART, CACHE = f'{BASE}/artifacts', f'{BASE}/archives'
os.makedirs(WORK, exist_ok=True); sys.path.insert(0, os.getcwd())
assert os.path.exists(f'{ART}/splits.json'), f'{ART}/splits.json missing, run N0 first'

REPO = f'{WORK}/deeparuco'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/AVAuco/deeparuco.git',REPO],check=True)
sys.path.insert(0, REPO)
from impl.aruco import id_to_bits
from deeparuco_port import load_weights, NumpyDecoder

DEV_ = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
splits = json.load(open(f'{ART}/splits.json'))
DEVV, HELDOUT, STRIDE = splits['dev'], splits['heldout'][0], splits['benchmark_stride']
Wk  = load_weights(f'{REPO}/models/dec_new.h5')
CB  = np.stack([np.asarray(id_to_bits(i),np.uint8).reshape(6,6) for i in range(250)])
ROT = np.stack([np.rot90(CB,k,axes=(1,2)).reshape(250,36) for k in range(4)]).astype(np.float32)

SHADOW = f'{WORK}/shadow'; os.makedirs(SHADOW, exist_ok=True)
for i in range(1,7):
    if not os.path.isdir(f'{SHADOW}/video_{i}'):
        tar = f'{CACHE}/video_{i}.tar.gz'
        assert os.path.exists(tar), f'{tar} missing, see the download cell in N0'
        subprocess.run(['tar','-xzf',tar,'-C',SHADOW], check=True)

N0M = json.load(open(f'{ART}/manifest.json'))
for vid in DEVV + [HELDOUT]:
    n = len(glob.glob(f'{SHADOW}/{vid}/*.png'))
    assert n == N0M['shadow_aruco']['frames'][vid], f'{vid}: {n} frames on disk, manifest says other'
print('torch', torch.__version__, '| device', DEV_)
if DEV_.type != 'cuda':
    print('\n*** THIS IS A CPU RUNTIME. One training run takes about 40 minutes here')
    print('*** and about 2 minutes on a GPU. Use Runtime, Change runtime type, T4 or L4,')
    print('*** then rerun every cell from the top before training.')
print('dev:', DEVV, '| held out:', HELDOUT, '| stride', STRIDE)


Mounted at /content/drive
torch 2.14.1+cu130 | device cuda
dev: ['video_1', 'video_2', 'video_3', 'video_4', 'video_5'] | held out: video_6 | stride 6


In [4]:
r_int = lambda v: int(round(v))
VALUES = {'grey':127, 'black':0, 'white':255}

def ordered_corners(xs, ys):
    cx, cy = np.mean(xs), np.mean(ys)
    idx = np.argsort(np.arctan2(xs-cx, ys-cy))
    return [v for i in idx for v in (xs[i], ys[i])]

def warp32(crop, c, t=32):
    dst = np.array([[0,0],[0,t-1],[t-1,t-1],[t-1,0]], np.float32)
    src = np.clip(np.array([[c[0]*63,c[1]*63],[c[2]*63,c[3]*63],
                            [c[4]*63,c[5]*63],[c[6]*63,c[7]*63]]),0,63).astype(np.float32)
    return cv2.warpPerspective(crop, cv2.getPerspectiveTransform(src,dst), (t,t))

def get_corners(m):
    c = m['corners']
    if isinstance(c[0], dict): return np.array([[p['x'],p['y']] for p in c], np.float32)
    return np.asarray(c, np.float32).reshape(4,2)

def cell_occlusion(m32):
    s = 32/8.0; out = np.zeros((6,6), bool)
    for r in range(6):
        for c in range(6):
            out[r,c] = m32[int((r+1)*s):int((r+2)*s), int((c+1)*s):int((c+2)*s)].mean() > 127.5
    return out

def prepare(img, mask, corners, scale=None):
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None, None
    sl = (slice(r_int(mny), r_int(mxy)), slice(r_int(mnx), r_int(mxx)))
    crop = cv2.resize(img[sl], (64,64)).astype(np.float32)
    lo, hi = (crop.min(), crop.max()) if scale is None else scale
    crop = np.clip((crop - lo)/(hi - lo + 1e-9), 0, 1)
    nc = ordered_corners((x-mnx)/(mxx-mnx), (y-mny)/(mxy-mny))
    m32 = warp32(cv2.resize(mask[sl],(64,64)), nc) if mask.any() else np.zeros((32,32), np.uint8)
    return warp32(crop, nc), cell_occlusion(m32)

def occluder_box(img, corners, level, position, key, mid):
    if level <= 0: return None
    x, y = corners[:,0], corners[:,1]
    x0,x1,y0,y1 = x.min(),x.max(),y.min(),y.max(); w,h = x1-x0, y1-y0
    if position == 'edge':
        side = int(np.random.default_rng(zlib.crc32(f'occlusion|{level}|{key}|{mid}'.encode())).integers(0,4))
        box = [(x0,y0,x0+level*w,y1),(x1-level*w,y0,x1,y1),
               (x0,y0,x1,y0+level*h),(x0,y1-level*h,x1,y1)][side]
    else:
        cx, cy = (x0+x1)/2, (y0+y1)/2
        box = (cx-level*w/2, cy-level*h/2, cx+level*w/2, cy+level*h/2)
    a,b,c,d = [int(round(v)) for v in box]
    a,b = max(a,0),max(b,0); c,d = min(c,img.shape[1]),min(d,img.shape[0])
    return (a,b,c,d) if (c>a and d>b) else None

def apply_occluder(img, box, value):
    if box is None: return img, np.zeros(img.shape[:2], np.uint8)
    if isinstance(value, str): value = VALUES[value]
    a,b,c,d = box
    out = img.copy(); mk = np.zeros(img.shape[:2], np.uint8)
    out[b:d,a:c] = int(value); mk[b:d,a:c] = 255
    return out, mk

def pairs_for(vid):
    out = []
    for f in sorted(glob.glob(f'{SHADOW}/{vid}/*.png')):
        j = f'{SHADOW}/{vid}/corrected_annotations/{os.path.splitext(os.path.basename(f))[0]}.json'
        if os.path.exists(j): out.append((f,j))
    return out

class SpatialPReLU(nn.Module):
    def __init__(s,c,h,w): super().__init__(); s.a = nn.Parameter(torch.zeros(c,h,w))
    def forward(s,x): return torch.where(x>0, x, s.a.unsqueeze(0)*x)

class Decoder(nn.Module):
    """8 and 4 filters, as in the released code. pool is 'max' or 'avg'."""
    def __init__(s, out_ch=1, pool='max'):
        super().__init__()
        s.c1=nn.Conv2d(1,8,5); s.p1=SpatialPReLU(8,28,28)
        s.c2=nn.Conv2d(8,4,3); s.p2=SpatialPReLU(4,12,12)
        s.c3=nn.Conv2d(4,out_ch,1); s.out_ch=out_ch
        s.pool = F.max_pool2d if pool=='max' else F.avg_pool2d
    def forward(s,x):
        x=s.pool(s.p1(s.c1(x)),2); x=s.pool(s.p2(s.c2(x)),2); x=s.c3(x)
        return (x[:,0], x[:,1]) if s.out_ch==2 else (x[:,0], None)
print('pipeline ready')


pipeline ready


In [5]:
def _conv(x,w,b):
    kh,kw,_,_ = w.shape
    return np.einsum('nohckl,klci->nohi', sliding_window_view(x,(kh,kw),axis=(1,2)), w, optimize=True)+b
def _pool(x):
    N,H,Wd,Cc = x.shape; return x.reshape(N,H//2,2,Wd//2,2,Cc).max(axis=(2,4))

def decode_published(x8, chunk=4096):
    out=[]
    for s in range(0,len(x8),chunk):
        x = x8[s:s+chunk].astype(np.float32)[...,None]/255.0
        x = _pool(np.where((y:=_conv(x,Wk['c1_w'],Wk['c1_b']))>0,y,Wk['p1_a']*y))
        x = _pool(np.where((y:=_conv(x,Wk['c2_w'],Wk['c2_b']))>0,y,Wk['p2_a']*y))
        out.append(1/(1+np.exp(-np.clip(_conv(x,Wk['c3_w'],Wk['c3_b'])[...,0],-60,60))))
    return np.concatenate(out).astype(np.float32)

def decode_torch(net, x8, chunk=8192):
    ps, es = [], []
    with torch.no_grad():
        for s in range(0,len(x8),chunk):
            xb = torch.tensor(x8[s:s+chunk], dtype=torch.float32).unsqueeze(1).to(DEV_)/255.0
            lo, el = net(xb)
            ps.append(torch.sigmoid(lo).cpu().numpy())
            es.append(torch.sigmoid(el).cpu().numpy() if el is not None else np.zeros((len(lo),6,6),np.float32))
    return np.concatenate(ps), np.concatenate(es)

def dec_soft(P, Wt=None):
    LP = np.log(np.clip(P,1e-6,1)).reshape(len(P),36)
    LQ = np.log(np.clip(1-P,1e-6,1)).reshape(len(P),36)
    if Wt is not None:
        w = Wt.reshape(len(P),36); LP, LQ = LP*w, LQ*w
    return np.stack([LP@ROT[k].T + LQ@(1-ROT[k]).T for k in range(4)]).max(0).argmax(1)
def hamming_w(P, W=None):
    B = (P.reshape(len(P),36) > 0.5).astype(np.float32)
    W = np.ones_like(B) if W is None else W.reshape(len(P),36).astype(np.float32)
    return np.stack([(B*W)@(1-ROT[k]).T + ((1-B)*W)@ROT[k].T for k in range(4)]).min(0)

def hard_expected(P, ids, W=None):
    """Exact expected accuracy with fair random tie breaking, per marker (0 to 1)."""
    D = hamming_w(P, W); tied = D == D.min(1, keepdims=True)
    return tied[np.arange(len(ids)), ids] / tied.sum(1)
print('decoders ready')


decoders ready


In [6]:
def paired(a_ok, b_ok, groups, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    uf, inv = np.unique(groups, return_inverse=True)
    by = [np.where(inv==i)[0] for i in range(len(uf))]
    obs = a_ok.mean() - b_ok.mean()
    d = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p] for p in rng.integers(0, len(by), len(by))])
        d[i] = a_ok[idx].mean() - b_ok[idx].mean()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2*min((d <= 0).mean(), (d >= 0).mean())
    return float(obs), float(lo), float(hi), max(float(p), 1/n_boot)
from scipy import stats as sst

def welch(a, b):
    """Unpaired difference of means across seeds with a Welch t interval."""
    a, b = np.asarray(a, float), np.asarray(b, float); m = a.mean() - b.mean()
    va, vb = a.var(ddof=1)/len(a), b.var(ddof=1)/len(b); se = np.sqrt(va + vb)
    if se == 0: return float(m), float(m), float(m)
    df = (va + vb)**2 / (va**2/(len(a)-1) + vb**2/(len(b)-1))
    h = sst.t.ppf(0.975, df) * se
    return float(m), float(m - h), float(m + h)
print('statistics ready')


statistics ready


In [7]:
# ---------------------------------------------------------------- Flying ArUco v2
FLY = f'{WORK}/flyingarucov2'
if not os.path.isdir(FLY):
    os.makedirs(FLY, exist_ok=True)
    tar = f'{CACHE}/flyingarucov2.tar.gz'
    assert os.path.exists(tar), f'{tar} missing, see N0'
    subprocess.run(['tar','-xzf',tar,'-C',FLY], check=True)
fly_json = sorted(glob.glob(f'{FLY}/**/*.json', recursive=True))
assert len(fly_json) == N0M['flying_aruco_v2']['images'], \
    f'{len(fly_json)} annotations on disk, N0 manifest says {N0M["flying_aruco_v2"]["images"]}'

def bits_for(mid, rot, conv):
    b = CB[mid]; k = rot if conv in (0,1) else -rot
    return np.rot90(b, k) if conv in (0,2) else np.rot90(b, -k)

ref = NumpyDecoder(Wk); probe = []
for jp in fly_json[:60]:
    img = cv2.imread(jp.replace('.json','.jpg'), cv2.IMREAD_GRAYSCALE)
    if img is None: continue
    for m in json.load(open(jp))['markers'][:4]:
        x32, _ = prepare(img, np.zeros(img.shape[:2], np.uint8), get_corners(m))
        if x32 is None: continue
        probe.append((ref(x32), int(m['id']), int(m.get('rot', m.get('rotation', 0)))))
assert len(probe) > 20, f'only {len(probe)} probe markers, Flying ArUco did not load'
scores = {c: float(np.mean([np.mean((p>0.5).astype(np.uint8)==bits_for(i,r,c)) for p,i,r in probe]))
          for c in range(4)}
CONV = max(scores, key=scores.get)
assert scores[CONV] > 0.90, f'no rotation convention agrees ({scores}), labels would be wrong'
print(f'{len(fly_json)} Flying ArUco v2 annotations | rotation convention {CONV} ({scores[CONV]:.4f})')

2500 Flying ArUco v2 annotations | rotation convention 0 (0.9912)


In [8]:
# ---------------------------------------------------------------- training crops
OCC_TRAIN = [0.0, 0.10, 0.20, 0.25, 0.35, 0.45, 0.55]
N_SAMPLES = 5

def build_training_set(value):
    """value is 127 for arms B, C, D, M, G, or 'random' for arm E."""
    X, Ybit, Yocc = [], [], []
    t0 = time.time()
    for n, jp in enumerate(fly_json):
        img = cv2.imread(jp.replace('.json','.jpg'), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        key = os.path.basename(jp)
        for m in json.load(open(jp))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            if mid >= 250: continue
            bits = bits_for(mid, int(m.get('rot', m.get('rotation', 0))), CONV)
            rng = np.random.default_rng(zlib.crc32(f'train|{key}|{mid}'.encode()))
            for L in rng.choice(OCC_TRAIN, N_SAMPLES, replace=False):
                v = int(rng.integers(0,256)) if value == 'random' else value
                box = occluder_box(img, cn, float(L), 'edge', key, mid)
                oi, om = apply_occluder(img, box, v)
                x32, occ = prepare(oi, om, cn)
                if x32 is None: continue
                X.append((x32*255).astype(np.uint8)); Ybit.append(bits); Yocc.append(occ)
        if n % 800 == 0: print(f'  {value}: {n}/{len(fly_json)} ({time.time()-t0:.0f}s) crops={len(X)}')
    X = np.stack(X); Ybit = np.stack(Ybit).astype(np.float32); Yocc = np.stack(Yocc).astype(np.float32)
    assert len(X) > 50000, f'only {len(X)} crops, expected about 64840'
    print(f'  {value}: {len(X)} crops, mean occluded cells {Yocc.sum((1,2)).mean():.2f}, '
          f'{time.time()-t0:.0f}s')
    return X, Ybit, Yocc

SETS = {127: build_training_set(127), 'random': build_training_set('random')}
print('\ntraining sets ready')

  127: 0/2500 (0s) crops=40
  127: 800/2500 (19s) crops=20830
  127: 1600/2500 (37s) crops=41645
  127: 2400/2500 (54s) crops=62275
  127: 64840 crops, mean occluded cells 7.00, 56s
  random: 0/2500 (0s) crops=40
  random: 800/2500 (18s) crops=20830
  random: 1600/2500 (36s) crops=41645
  random: 2400/2500 (54s) crops=62275
  random: 64840 crops, mean occluded cells 7.00, 56s

training sets ready


In [9]:
# =================================================================================
# Everything the training loop needs is defined in this one cell, so a runtime
# restart cannot leave half of it missing.
# =================================================================================
CKPT = f'{BASE}/ckpt_n14'; os.makedirs(CKPT, exist_ok=True)
CODE_TAG = 'n14-v1'         # change this whenever the training code changes

SEEDS   = [0, 1, 2, 3, 4]
MARGIN  = 1.0               # equivalence margin, in accuracy points
EPOCHS, PATIENCE, BS = 200, 20, 256
ALLOW_CPU = False           # set True only if you accept about 40 minutes per run
ALLOW_NONDETERMINISM = False

# arm -> output channels, loss masking, pooling, training occluder brightness
ARMS = {
 'B': dict(out_ch=1, mask_bits=False, pool='max', value=127),
 'C': dict(out_ch=2, mask_bits=True,  pool='max', value=127),
 'D': dict(out_ch=2, mask_bits=False, pool='max', value=127),
 'E': dict(out_ch=1, mask_bits=False, pool='max', value='random'),
 'F': dict(out_ch=2, mask_bits=True,  pool='max', value='random'),
 'M': dict(out_ch=1, mask_bits=False, pool='max', value=127),
 'G': dict(out_ch=1, mask_bits=False, pool='avg', value=127),
}
assert DEV_.type == 'cuda' or ALLOW_CPU, (
    'This is a CPU runtime and 35 runs would take about a day. '
    'Switch to a GPU (Runtime, Change runtime type), rerun every cell from the top, '
    'or set ALLOW_CPU = True if you really mean it.')

GPU_SETS = {}
for v, (X, Yb, Yo) in SETS.items():
    GPU_SETS[v] = (torch.from_numpy(X).to(DEV_),                 # uint8 (N,32,32)
                   torch.from_numpy(Yb).to(DEV_),                # float32 (N,6,6)
                   torch.from_numpy(Yo).to(DEV_))
    print(f'  set {v}: {X.nbytes/1e6:.0f} MB of crops moved to {DEV_}')

SPLIT = {}
for v, (X, _, _) in SETS.items():
    idx = np.random.default_rng(42).permutation(len(X)); cut = int(0.9*len(X))
    SPLIT[v] = (torch.from_numpy(idx[:cut]).to(DEV_), torch.from_numpy(idx[cut:]).to(DEV_))

def signature(arm):
    return json.dumps(dict(ARMS[arm], code=CODE_TAG, epochs=EPOCHS, patience=PATIENCE, bs=BS),
                      sort_keys=True, default=str)

def train_arm(arm, seed, epochs=EPOCHS, patience=PATIENCE, save=True):
    cfg = ARMS[arm]
    Xg, Bg, Og = GPU_SETS[cfg['value']]
    tr, va = SPLIT[cfg['value']]
    path = f'{CKPT}/arm_{arm}_s{seed}.pt'
    if save and os.path.exists(path):
        ck = torch.load(path, map_location=DEV_)
        assert isinstance(ck, dict) and ck.get('sig') == signature(arm), \
            f'{path} was made with other settings or code. Delete it and rerun.'
        net = Decoder(cfg['out_ch'], cfg['pool']).to(DEV_)
        net.load_state_dict(ck['state']); net.eval()
        return net, None, 0.0
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    rng = np.random.default_rng(seed)
    gen = torch.Generator(device=DEV_); gen.manual_seed(seed)
    net = Decoder(cfg['out_ch'], cfg['pool']).to(DEV_)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=8)

    def run(ids, train):
        net.train(train)
        order = ids[torch.randperm(len(ids), device=DEV_, generator=gen)] if train else ids
        tot = torch.zeros((), device=DEV_); n = 0
        with torch.set_grad_enabled(train):
            for s in range(0, len(order), BS):
                b  = order[s:s+BS]
                xb = Xg[b].unsqueeze(1).float()/255.0
                bb = Bg[b]; ob = Og[b]
                if train:                                    # same augmentation as N3b
                    k, fl = int(rng.integers(0,4)), int(rng.integers(0,3))
                    if k:
                        xb = torch.rot90(xb,k,dims=(-2,-1))
                        bb = torch.rot90(bb,k,dims=(-2,-1)); ob = torch.rot90(ob,k,dims=(-2,-1))
                    if fl == 1:
                        xb, bb, ob = torch.flip(xb,(-1,)), torch.flip(bb,(-1,)), torch.flip(ob,(-1,))
                    elif fl == 2:
                        xb, bb, ob = torch.flip(xb,(-2,)), torch.flip(bb,(-2,)), torch.flip(ob,(-2,))
                logit, elogit = net(xb)
                if cfg['mask_bits']:
                    vis  = 1.0 - ob
                    loss = (F.binary_cross_entropy_with_logits(logit, bb, reduction='none')*vis
                            ).sum()/vis.sum().clamp(min=1)
                else:
                    loss = F.binary_cross_entropy_with_logits(logit, bb)
                if cfg['out_ch'] == 2:
                    loss = loss + F.binary_cross_entropy_with_logits(elogit, ob)
                if train:
                    opt.zero_grad(); loss.backward(); opt.step()
                tot = tot + loss.detach()*len(b); n += len(b)
        return (tot/n).item()            # one sync per epoch, not one per batch

    best, bad, t0, state, ep = 1e9, 0, time.time(), None, 0
    for ep in range(epochs):
        run(tr, True); vl = run(va, False); sch.step(vl)
        if vl < best - 1e-5:
            best, bad = vl, 0
            state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval()
    if save:                                             # written only once the run is complete
        tmp = path + '.tmp'
        torch.save(dict(state=state, sig=signature(arm), best=best, epochs=ep+1), tmp)
        os.replace(tmp, path)
    return net, best, time.time()-t0

# ----- determinism probe: the same short run twice must give identical weights
netA, _, secs = train_arm('B', SEEDS[0], epochs=3, patience=99, save=False)
netB, _, _    = train_arm('B', SEEDS[0], epochs=3, patience=99, save=False)
diff = max(float((a-b).abs().max()) for a, b in zip(netA.state_dict().values(), netB.state_dict().values()))
print(f'\ndeterminism probe: same seed twice, largest weight difference {diff:.2e}')
assert diff == 0.0 or ALLOW_NONDETERMINISM, (
    'Training is still not deterministic. Restart the session (Runtime, Restart session) and '
    'Run all, so the setting is applied before torch starts. If it still fails, set '
    'ALLOW_NONDETERMINISM = True and report the B against M gap as run to run noise.')
per_epoch = secs/3
print(f'probe: 3 epochs of arm B took {secs:.1f}s, so {per_epoch:.2f}s per epoch')
print(f'projected: about {per_epoch*150/60:.1f} min per run, '
      f'{per_epoch*150*len(ARMS)*len(SEEDS)/3600:.1f} hours for all {len(ARMS)*len(SEEDS)} runs')
assert per_epoch < 5, 'still too slow, stop and check the device before continuing'


  set 127: 66 MB of crops moved to cuda
  set random: 66 MB of crops moved to cuda

determinism probe: same seed twice, largest weight difference 0.00e+00
probe: 3 epochs of arm B took 6.6s, so 2.21s per epoch
projected: about 5.5 min per run, 3.2 hours for all 35 runs


In [10]:
MODELS, TIMES = {}, {}
total = len(ARMS)*len(SEEDS); k = 0
t_all = time.time()
for arm in ARMS:
    for s in SEEDS:
        k += 1
        net, vl, secs = train_arm(arm, s)
        MODELS[(arm,s)] = net; TIMES[(arm,s)] = secs
        tag = 'loaded from checkpoint' if vl is None else f'val {vl:.4f} ({secs:.0f}s)'
        print(f'  [{k:>2}/{total}] arm {arm} seed {s}: {tag}', flush=True)
print(f'\nall {total} runs done in {(time.time()-t_all)/60:.1f} min')

  [ 1/35] arm B seed 0: val 0.1615 (95s)
  [ 2/35] arm B seed 1: val 0.1510 (61s)
  [ 3/35] arm B seed 2: val 0.1565 (104s)
  [ 4/35] arm B seed 3: val 0.1516 (118s)
  [ 5/35] arm B seed 4: val 0.1516 (92s)
  [ 6/35] arm C seed 0: val 0.0586 (104s)
  [ 7/35] arm C seed 1: val 0.0715 (137s)
  [ 8/35] arm C seed 2: val 0.0808 (138s)
  [ 9/35] arm C seed 3: val 0.0569 (140s)
  [10/35] arm C seed 4: val 0.0608 (101s)
  [11/35] arm D seed 0: val 0.1845 (99s)
  [12/35] arm D seed 1: val 0.1814 (128s)
  [13/35] arm D seed 2: val 0.2069 (124s)
  [14/35] arm D seed 3: val 0.1830 (150s)
  [15/35] arm D seed 4: val 0.1800 (132s)
  [16/35] arm E seed 0: val 0.1670 (93s)
  [17/35] arm E seed 1: val 0.1579 (62s)
  [18/35] arm E seed 2: val 0.1616 (103s)
  [19/35] arm E seed 3: val 0.1564 (118s)
  [20/35] arm E seed 4: val 0.1547 (91s)
  [21/35] arm F seed 0: val 0.0734 (120s)
  [22/35] arm F seed 1: val 0.0715 (157s)
  [23/35] arm F seed 2: val 0.1008 (157s)
  [24/35] arm F seed 3: val 0.0683 (157s)

In [11]:
TARGETS  = [3, 6, 9, 12, 15, 18]
COLOURS  = ['grey', 'white', 'black']
POSITION = 'edge'

cal = []
for vid in DEVV[:2]:
    for f, j in pairs_for(vid)[::STRIDE*8]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cal.append((img, get_corners(m), key, int(m['id'])))
        if len(cal) > 400: break
    if len(cal) > 400: break
def mean_cells(level):
    tot=n=0
    for img, cn, key, mid in cal:
        oi, om = apply_occluder(img, occluder_box(img, cn, level, POSITION, key, mid), 'grey')
        _, occ = prepare(oi, om, cn)
        if occ is None: continue
        tot += occ.sum(); n += 1
    return tot/max(n,1)
curve = [(float(L), mean_cells(float(L))) for L in np.arange(0.02, 0.70, 0.02)]
LEVEL_FOR = {T: min(curve, key=lambda lc: abs(lc[1]-T))[0] for T in TARGETS}
print('severity calibration:', {T: round(LEVEL_FOR[T],2) for T in TARGETS})

CLEAN_KEY = ('clean', 0)
CONDS = [(c,T) for c in COLOURS for T in TARGETS]
EV = {k: {'x':[], 'id':[], 'frame':[], 'occ':[], 'mask':[]} for k in [CLEAN_KEY] + CONDS}
def _add(k, x32, occ, mid, key):
    d = EV[k]
    d['x'].append((x32*255).astype(np.uint8)); d['id'].append(mid)
    d['frame'].append(key); d['occ'].append(int(occ.sum())); d['mask'].append(occ)

t0 = time.time()
for vid in DEVV:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        zero = np.zeros(img.shape[:2], np.uint8)
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            xc, oc = prepare(img, zero, cn)
            if xc is None: continue                      # same corners, so every condition would fail too
            _add(CLEAN_KEY, xc, oc, mid, key)
            for T in TARGETS:
                box = occluder_box(img, cn, LEVEL_FOR[T], POSITION, key, mid)
                for col in COLOURS:
                    oi, om = apply_occluder(img, box, col)
                    x32, occ = prepare(oi, om, cn)
                    _add((col,T), x32, occ, mid, key)
    print(f'  {vid} ({time.time()-t0:.0f}s)')
for k in EV: EV[k] = {kk: np.array(vv) for kk, vv in EV[k].items()}
n = len(EV[CLEAN_KEY]['id'])
assert all(len(EV[k]['id']) == n for k in EV), 'conditions are misaligned'
print(f'\neval set: {n} markers per condition, plus a clean condition')
assert n > 1500, f'only {n} markers, data did not load correctly'


severity calibration: {3: 0.24, 6: 0.3, 9: 0.34, 12: 0.4, 15: 0.46, 18: 0.5}
  video_1 (45s)
  video_2 (79s)
  video_3 (127s)
  video_4 (168s)
  video_5 (201s)

eval set: 12276 markers per condition, plus a clean condition


In [12]:
ACC = {}            # (arm, seed, colour, target) -> per marker correctness
for (arm, s), net in MODELS.items():
    for col in COLOURS:
        for T in TARGETS:
            d = EV[(col,T)]
            P, _ = decode_torch(net, d['x'])
            ACC[(arm,s,col,T)] = (dec_soft(P) == d['id'])
print('evaluated', len(ACC), 'arm and condition combinations\n')

def summary(arm, col, T):
    v = np.array([ACC[(arm,s,col,T)].mean() for s in SEEDS])*100
    return v.mean(), v.std(ddof=1)

print('ARM ACCURACY, mean and sd over', len(SEEDS), 'seeds, development data\n')
for col in COLOURS:
    print(f'--- occluder brightness: {col}')
    print(f"{'cells':>7}" + ''.join(f'{a:>16}' for a in ARMS))
    for T in TARGETS:
        line = f'{EV[(col,T)]["occ"].mean():>7.1f}'
        for arm in ARMS:
            m, sd = summary(arm, col, T); line += f'{m:>10.2f}+-{sd:<4.2f}'
        print(line)
    print()

evaluated 630 arm and condition combinations

ARM ACCURACY, mean and sd over 5 seeds, development data

--- occluder brightness: grey
  cells               B               C               D               E               F               M               G
    3.2     97.21+-2.19     88.64+-4.53     95.36+-2.00     97.47+-1.54     86.80+-1.44     97.21+-2.19     98.84+-0.25
    5.7     94.87+-2.96     78.96+-7.21     91.76+-2.25     95.88+-1.69     74.56+-3.62     94.87+-2.96     96.50+-1.12
    8.5     92.56+-3.49     72.65+-8.57     89.81+-3.01     93.82+-2.19     66.06+-5.28     92.56+-3.49     94.80+-0.86
   11.5     90.23+-3.61     63.30+-8.98     86.84+-2.69     91.51+-2.88     55.01+-5.74     90.23+-3.61     92.18+-1.69
   15.6     85.82+-3.35     52.82+-10.35     82.09+-3.23     87.38+-2.53     41.83+-7.12     85.82+-3.35     87.54+-1.86
   18.0     80.11+-3.71     41.28+-9.99     76.05+-3.48     82.06+-3.42     30.65+-6.67     80.11+-3.71     80.76+-2.51

--- occluder brightness:

## Determinism check

B and M have identical settings and seeds. With deterministic training their weights must be
identical. If they are, every number below is reproducible from the seed alone.


In [13]:
worst_w = worst_a = 0.0
for s in SEEDS:
    sb, sm = MODELS[('B',s)].state_dict(), MODELS[('M',s)].state_dict()
    dw = max(float((sb[k]-sm[k]).abs().max()) for k in sb)
    da = max(abs(100*(ACC[('B',s,c,T)].mean() - ACC[('M',s,c,T)].mean())) for c in COLOURS for T in TARGETS)
    worst_w, worst_a = max(worst_w, dw), max(worst_a, da)
    print(f'  seed {s}: largest weight difference {dw:.2e}, largest accuracy gap {da:.2f} points')
print()
if worst_w == 0.0:
    print('Deterministic: B and M are identical. Results are reproducible from the seed.')
else:
    print(f'WARNING: B and M differ (weights {worst_w:.2e}, accuracy {worst_a:.2f} points). '
          'Training is not fully deterministic, so treat this gap as run to run noise.')


  seed 0: largest weight difference 0.00e+00, largest accuracy gap 0.00 points
  seed 1: largest weight difference 0.00e+00, largest accuracy gap 0.00 points
  seed 2: largest weight difference 0.00e+00, largest accuracy gap 0.00 points
  seed 3: largest weight difference 0.00e+00, largest accuracy gap 0.00 points
  seed 4: largest weight difference 0.00e+00, largest accuracy gap 0.00 points

Deterministic: B and M are identical. Results are reproducible from the seed.


## Unified evaluation

Every arm plus the published decoder (P), on the clean condition and on grey, white and black:

* `soft`: soft log likelihood
* `hard`: nearest codeword with exact fair tie breaking
* `soft_erasure`: soft decoding weighted by 1 minus the erasure output (C, D and F)
* `soft_oracle`, `hard_oracle`: hidden cells removed using the true mask, an upper bound

Then the grey minus black gap, and how far each arm's best practical decoder falls short of
its own oracle.


In [14]:
DECODERS = ['soft', 'hard', 'soft_erasure', 'soft_oracle', 'hard_oracle']

def score(P, E, d):
    ids = d['id']; vis = 1.0 - d['mask'].astype(np.float32)
    out = {'soft':        (dec_soft(P) == ids).astype(np.float32),
           'hard':        hard_expected(P, ids),
           'soft_oracle': (dec_soft(P, vis) == ids).astype(np.float32),
           'hard_oracle': hard_expected(P, ids, vis)}
    if E is not None:
        out['soft_erasure'] = (dec_soft(P, 1 - E) == ids).astype(np.float32)
    return out

R = {}; t0 = time.time()
for k, d in EV.items():
    for dec, v in score(decode_published(d['x']), None, d).items():
        R[('P','pub') + k + (dec,)] = v
for (arm, s), net in MODELS.items():
    for k, d in EV.items():
        P, E = decode_torch(net, d['x'])
        for dec, v in score(P, E if ARMS[arm]['out_ch'] == 2 else None, d).items():
            R[(arm, s) + k + (dec,)] = v
print(f'scored {len(R)} combinations in {time.time()-t0:.0f}s\n')

ARM_SEEDS = {'P': ['pub'], **{a: SEEDS for a in ARMS}}
def seed_values(arm, col, T, dec):
    return [100*R[(arm,s,col,T,dec)].mean() for s in ARM_SEEDS[arm] if (arm,s,col,T,dec) in R]
def stat(arm, col, T, dec):
    v = seed_values(arm, col, T, dec)
    if not v: return None
    return float(np.mean(v)), (float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)
def practical(arm):
    return 'soft_erasure' if arm in ARMS and ARMS[arm]['out_ch'] == 2 else 'soft'

print('CLEAN, no occluder (domain shift check)\n')
for arm in ARM_SEEDS:
    a, b = stat(arm,'clean',0,'soft'), stat(arm,'clean',0,'hard')
    line = f'{arm:>4}   soft {a[0]:6.2f}+-{a[1]:<4.2f}   hard {b[0]:6.2f}+-{b[1]:<4.2f}'
    if practical(arm) == 'soft_erasure':
        e = stat(arm,'clean',0,'soft_erasure'); line += f'   soft_erasure {e[0]:6.2f}+-{e[1]:<4.2f}'
    print(line)

print('\nACCURACY BY ARM AND DECODER, mean +- sd over seeds (P is the published decoder, one run)\n')
for col in COLOURS:
    print(f'--- {col}')
    print(f"{'arm':>4}{'decoder':>14}" + ''.join(f'{EV[(col,T)]["occ"].mean():>13.1f}' for T in TARGETS))
    for arm in ARM_SEEDS:
        for dec in DECODERS:
            if stat(arm, col, TARGETS[0], dec) is None: continue
            print(f'{arm:>4}{dec:>14}' + ''.join(
                f'{m:>8.2f}+-{sd:<3.1f}' for m, sd in (stat(arm, col, T, dec) for T in TARGETS)))
        print()

print('GREY MINUS BLACK, best practical decoder per arm (positive means black is harder)\n')
for arm in ARM_SEEDS:
    dec = practical(arm)
    print(f'{arm:>4}{dec:>14}' + ''.join(f'{stat(arm,"grey",T,dec)[0]-stat(arm,"black",T,dec)[0]:>9.2f}'
                                         for T in TARGETS))

print('\nGAP TO ORACLE: soft_oracle minus best practical decoder (small means the arm already')
print('knows which cells are hidden)\n')
for col in COLOURS:
    print(f'--- {col}')
    for arm in ARM_SEEDS:
        dec = practical(arm)
        print(f'{arm:>4}{dec:>14}' + ''.join(
            f'{stat(arm,col,T,"soft_oracle")[0]-stat(arm,col,T,dec)[0]:>9.2f}' for T in TARGETS))
    print()

out = {f'{a}|{s}|{c}|{T}|{dec}': float(v.mean()) for (a,s,c,T,dec), v in R.items()}
with open(f'{ART}/n14_unified_dev.json','w') as fh: json.dump(out, fh, indent=1)
print('saved', f'{ART}/n14_unified_dev.json')


scored 3021 combinations in 384s

CLEAN, no occluder (domain shift check)

   P   soft  99.54+-0.00   hard  99.48+-0.00
   B   soft  99.60+-0.59   hard  98.96+-1.33
   C   soft  99.75+-0.09   hard  99.17+-0.48   soft_erasure  99.74+-0.09
   D   soft  99.25+-0.50   hard  98.07+-1.27   soft_erasure  99.19+-0.49
   E   soft  99.82+-0.15   hard  99.31+-0.62
   F   soft  99.87+-0.06   hard  99.40+-0.32   soft_erasure  99.84+-0.09
   M   soft  99.60+-0.59   hard  98.96+-1.33
   G   soft  99.86+-0.09   hard  99.49+-0.22

ACCURACY BY ARM AND DECODER, mean +- sd over seeds (P is the published decoder, one run)

--- grey
 arm       decoder          3.2          5.7          8.5         11.5         15.6         18.0
   P          soft   85.70+-0.0   76.33+-0.0   65.40+-0.0   49.22+-0.0   31.97+-0.0   22.59+-0.0
   P          hard   89.03+-0.0   79.43+-0.0   69.45+-0.0   52.54+-0.0   34.67+-0.0   23.35+-0.0
   P   soft_oracle   93.16+-0.0   86.19+-0.0   78.51+-0.0   66.97+-0.0   54.37+-0.0   43.6

## Comparisons across seeds

Each arm is scored with the decoder it was designed for. Seeds of different arms are
independent runs, so the interval is Welch's unpaired t interval. `differs` means the
interval excludes zero, `equivalent` means it lies entirely inside plus or minus 1 point,
anything else is `inconclusive` and must never be written as "no effect".


In [15]:
COMPARE = [
 ('F','soft_erasure','C','soft_erasure','does random brightness training fix the erasure arm'),
 ('F','soft_erasure','E','soft',        'erasure arm against random brightness arm'),
 ('C','soft_erasure','B','soft',        'masked loss plus erasure output against baseline'),
 ('D','soft_erasure','B','soft',        'erasure output without masking against baseline'),
 ('E','soft',        'B','soft',        'random brightness training'),
 ('M','soft',        'G','soft',        'max against average pooling'),
]
CMP = []
for a1, d1, a2, d2, q in COMPARE:
    print(f'=== {a1} ({d1}) minus {a2} ({d2}): {q}')
    print(f"{'col':>7}{'cells':>7}{'diff':>9}{'95% CI':>20}  verdict")
    for col in COLOURS:
        for T in TARGETS:
            m, lo, hi = welch(seed_values(a1,col,T,d1), seed_values(a2,col,T,d2))
            if lo > 0 or hi < 0:                 v = 'differs'
            elif lo > -MARGIN and hi < MARGIN:   v = f'equivalent (within {MARGIN})'
            else:                                v = 'inconclusive'
            CMP.append(dict(a1=a1, d1=d1, a2=a2, d2=d2, colour=col, target=T,
                            cells=float(EV[(col,T)]['occ'].mean()), diff=m, lo=lo, hi=hi, verdict=v))
            print(f'{col:>7}{EV[(col,T)]["occ"].mean():>7.1f}{m:>9.2f}   [{lo:>6.2f},{hi:>6.2f}]  {v}')
    print()
print('A claim of no effect is only allowed where the verdict says equivalent.')


=== F (soft_erasure) minus C (soft_erasure): does random brightness training fix the erasure arm
    col  cells     diff              95% CI  verdict
   grey    3.2    -0.97   [ -2.87,  0.93]  inconclusive
   grey    5.7    -1.37   [ -3.68,  0.95]  inconclusive
   grey    8.5    -0.53   [ -3.26,  2.21]  inconclusive
   grey   11.5    -1.17   [ -4.36,  2.02]  inconclusive
   grey   15.6     0.64   [ -4.00,  5.29]  inconclusive
   grey   18.0     1.35   [ -3.39,  6.08]  inconclusive
  white    3.2    -0.64   [ -5.43,  4.15]  inconclusive
  white    5.7    -1.04   [ -5.89,  3.81]  inconclusive
  white    8.5     0.04   [ -5.29,  5.36]  inconclusive
  white   11.5    -0.38   [ -5.87,  5.10]  inconclusive
  white   15.6     1.88   [ -4.91,  8.67]  inconclusive
  white   18.0     2.34   [ -3.69,  8.38]  inconclusive
  black    3.2    -0.75   [ -5.00,  3.50]  inconclusive
  black    5.7    -0.63   [ -8.23,  6.98]  inconclusive
  black    8.5    -0.24   [-12.27, 11.79]  inconclusive
  black   

## Confidence cap for soft decoding

Probabilities are clipped to [eps, 1 minus eps] before soft decoding. eps = 0 is plain soft
decoding; as eps grows towards 0.5 it behaves more and more like hard decoding, with the
remaining confidence used to break ties. eps is chosen on videos 1 and 2 and every number in
the tables is from videos 3 to 5 only, so the choice is not tuned on the data it is judged on.
Done for the published decoder (P) and arm B.


In [16]:
EPS = [0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4]
TUNE_V = DEVV[:2]
vid_of = np.array([fr.split('/')[0] for fr in EV[CLEAN_KEY]['frame']])
tune = np.isin(vid_of, TUNE_V); test = ~tune
print(f'tuning on {TUNE_V} ({tune.sum()} markers), testing on the other videos ({test.sum()} markers)\n')

def soft_capped(P, eps):
    return dec_soft(np.clip(P, eps, 1-eps) if eps > 0 else P)

def cap_eval(get_P, seeds):
    out = {}
    for s in seeds:
        for k in [CLEAN_KEY] + CONDS:
            P = get_P(s, k); ids = EV[k]['id']
            for eps in EPS:
                ok = soft_capped(P, eps) == ids
                out[(s,eps,k,'tune')] = float(ok[tune].mean()); out[(s,eps,k,'test')] = float(ok[test].mean())
            h = hard_expected(P, ids)
            out[(s,'hard',k,'tune')] = float(h[tune].mean()); out[(s,'hard',k,'test')] = float(h[test].mean())
    return out

t0 = time.time()
CAP = {'P': cap_eval(lambda s, k: decode_published(EV[k]['x']), ['pub']),
       'B': cap_eval(lambda s, k: decode_torch(MODELS[('B',s)], EV[k]['x'])[0], SEEDS)}
print(f'done in {time.time()-t0:.0f}s\n')

CAP_CHOICE = {}
for arm, out in CAP.items():
    seeds = ARM_SEEDS[arm]
    tune_score = {eps: np.mean([out[(s,eps,k,'tune')] for s in seeds for k in CONDS]) for eps in EPS}
    best = max(tune_score, key=tune_score.get); CAP_CHOICE[arm] = best
    print(f'arm {arm}: mean tuning accuracy by eps  ' +
          '  '.join(f'{e}:{100*v:.2f}' for e, v in tune_score.items()))
    print(f'arm {arm}: chosen eps = {best}\n')
    print(f"{'col':>6}{'cells':>7}{'soft':>9}{'capped':>9}{'hard':>9}{'capped-soft':>13}{'capped-hard':>13}")
    for k in [CLEAN_KEY] + CONDS:
        g = lambda key: 100*np.mean([out[(s,key,k,'test')] for s in seeds])
        print(f"{k[0]:>6}{EV[k]['occ'].mean():>7.1f}{g(0.0):>9.2f}{g(best):>9.2f}{g('hard'):>9.2f}"
              f"{g(best)-g(0.0):>13.2f}{g(best)-g('hard'):>13.2f}")
    print()


tuning on ['video_1', 'video_2'] (4938 markers), testing on the other videos (7338 markers)

done in 195s

arm P: mean tuning accuracy by eps  0.0:58.91  0.01:59.04  0.02:57.15  0.05:58.48  0.1:57.42  0.15:57.54  0.2:55.05  0.3:55.65  0.4:57.08
arm P: chosen eps = 0.01

   col  cells     soft   capped     hard  capped-soft  capped-hard
 clean    0.0    99.74    99.74    99.71         0.00         0.03
  grey    3.2    85.73    88.65    89.60         2.92        -0.95
  grey    5.7    74.31    76.68    78.28         2.37        -1.60
  grey    8.5    64.99    69.77    69.46         4.78         0.32
  grey   11.5    47.17    50.87    50.84         3.71         0.04
  grey   15.6    31.89    33.40    35.06         1.51        -1.65
  grey   18.0    22.09    22.81    22.63         0.72         0.18
 white    3.2    76.15    84.23    86.32         8.08        -2.09
 white    5.7    60.78    67.16    69.54         6.38        -2.38
 white    8.5    50.07    57.94    59.10         7.88      

## Where the visible damage happens

Visible cells are split by distance from the nearest hidden cell: touching it (including
diagonals), one cell further, and further than that. If damage is concentrated next to the
occluder, it spills through the network's field of view. If it is spread evenly, it comes
from something global such as the border cue or the brightness scaling. The clean bit
accuracy is the reference.


In [17]:
import warnings; warnings.filterwarnings('ignore', message='Mean of empty slice')
# true bits per marker in the decoders' frame, recovered from the clean crops
Pc = decode_published(EV[CLEAN_KEY]['x']).reshape(-1, 36)
ids_c = EV[CLEAN_KEY]['id']
LPc, LQc = np.log(np.clip(Pc,1e-6,1)), np.log(np.clip(1-Pc,1e-6,1))
rot_score = np.stack([(LPc*ROT[k][ids_c]).sum(1) + (LQc*(1-ROT[k][ids_c])).sum(1) for k in range(4)])
TRUTH = ROT[rot_score.argmax(0), ids_c] > 0.5
print(f'clean bit accuracy, published decoder: {100*((Pc>0.5)==TRUTH).mean():.2f}%')
assert ((Pc>0.5)==TRUTH).mean() > 0.97, 'orientation recovery failed'

def dilate(m):
    p = np.pad(m, ((0,0),(1,1),(1,1))); out = np.zeros_like(m)
    for dy in range(3):
        for dx in range(3): out |= p[:, dy:dy+6, dx:dx+6]
    return out

def rings(M):
    d1 = dilate(M); d2 = dilate(d1)
    return {'touching': d1 & ~M, 'one away': d2 & ~d1, 'further': ~d2}

def probs(arm, s, k):
    if arm == 'P': return decode_published(EV[k]['x']).reshape(-1, 36), None
    P, E = decode_torch(MODELS[(arm,s)], EV[k]['x'])
    return P.reshape(-1, 36), (E.reshape(-1, 36) if ARMS[arm]['out_ch'] == 2 else None)

clean_ref = {}
for arm in ['P', 'B']:
    clean_ref[arm] = 100*np.mean([((probs(arm,s,CLEAN_KEY)[0] > 0.5) == TRUTH).mean() for s in ARM_SEEDS[arm]])
    print(f'clean bit accuracy, arm {arm}: {clean_ref[arm]:.2f}%')

DAMAGE = []
print(f"\n{'col':>6}{'cells':>7}{'ring':>10}{'share':>8}{'P acc':>9}{'P wrong>0.9':>13}{'B acc':>9}{'B wrong>0.9':>13}")
for col in COLOURS:
    for T in [6, 12, 18]:
        k = (col,T); rg = rings(EV[k]['mask'])
        res = {}
        for arm in ['P', 'B']:
            for s in ARM_SEEDS[arm]:
                P, _ = probs(arm, s, k)
                correct = (P > 0.5) == TRUTH; conf = np.maximum(P, 1-P)
                for name, sel in rg.items():
                    sel = sel.reshape(-1, 36)
                    acc = float(correct[sel].mean()) if sel.any() else float('nan')
                    cw  = float(((~correct) & (conf > 0.9))[sel].mean()) if sel.any() else float('nan')
                    res.setdefault((arm,name), []).append((acc, cw))
                    DAMAGE.append(dict(arm=arm, seed=s, colour=col, target=T, ring=name,
                                       n=int(sel.sum()), bit_acc=acc, conf_wrong=cw))
        for name, sel in rg.items():
            share = sel.mean()*36/max(1, 36 - EV[k]['mask'].reshape(-1,36).sum(1).mean())
            pa, pw = np.nanmean([r[0] for r in res[('P',name)]]), np.nanmean([r[1] for r in res[('P',name)]])
            ba, bw = np.nanmean([r[0] for r in res[('B',name)]]), np.nanmean([r[1] for r in res[('B',name)]])
            print(f'{col:>6}{EV[k]["occ"].mean():>7.1f}{name:>10}{100*share:>7.1f}%'
                  f'{100*pa:>9.2f}{100*pw:>12.2f}%{100*ba:>9.2f}{100*bw:>12.2f}%')
    print()
print('share = fraction of the visible cells that fall in that ring')


clean bit accuracy, published decoder: 97.61%
clean bit accuracy, arm P: 97.61%
clean bit accuracy, arm B: 97.43%

   col  cells      ring   share    P acc  P wrong>0.9    B acc  B wrong>0.9
  grey    5.7  touching   21.1%    65.13       34.57%    85.82        5.65%
  grey    5.7  one away   24.0%    89.21       10.32%    96.58        0.61%
  grey    5.7   further   54.9%    97.46        2.34%    97.59        0.43%
  grey   11.5  touching   30.7%    67.38       32.33%    85.75        5.83%
  grey   11.5  one away   30.8%    91.72        7.79%    97.11        0.51%
  grey   11.5   further   38.6%    97.66        2.14%    97.68        0.44%
  grey   18.0  touching   42.6%    69.28       30.38%    85.50        6.84%
  grey   18.0  one away   36.7%    92.79        6.75%    97.49        0.41%
  grey   18.0   further   20.7%    97.81        2.01%    97.63        0.48%

 white    5.7  touching   21.1%    57.74       42.06%    81.12        8.73%
 white    5.7  one away   24.0%    85.39       1

## Calibration of retrained arms

For hidden and visible cells: bit accuracy, mean confidence, share wrong while over 90% sure,
and mean negative log probability of the true bit. For arms with an erasure output, `flag`
is the share of cells it marks as hidden (above 0.5). On hidden cells that is the detection
rate, on visible cells it is the false alarm rate. The key question: does arm F flag hidden
black cells, which arm C cannot?


In [18]:
import warnings; warnings.filterwarnings('ignore', message='Mean of empty slice')
CAL_ARMS = ['P', 'B', 'C', 'E', 'F']
CAL_ROWS = []
for col in COLOURS:
    for T in [6, 12, 18]:
        k = (col,T); M = EV[k]['mask'].reshape(-1, 36)
        for arm in CAL_ARMS:
            for s in ARM_SEEDS[arm]:
                P, E = probs(arm, s, k)
                correct = (P > 0.5) == TRUTH; conf = np.maximum(P, 1-P)
                p_true = np.where(TRUTH, P, 1-P)
                for region, sel in (('hidden', M), ('visible', ~M)):
                    CAL_ROWS.append(dict(arm=arm, seed=s, colour=col, target=T, region=region,
                        bit_acc=float(correct[sel].mean()), conf=float(conf[sel].mean()),
                        conf_wrong=float(((~correct) & (conf > 0.9))[sel].mean()),
                        nll=float(-np.log(np.clip(p_true[sel],1e-6,1)).mean()),
                        flag=float((E[sel] > 0.5).mean()) if E is not None else float('nan')))

def agg(arm, col, T, region, key):
    return np.nanmean([r[key] for r in CAL_ROWS if r['arm']==arm and r['colour']==col
                       and r['target']==T and r['region']==region])

hdr = f"{'acc':>11}{'conf':>7}{'wrong>.9':>9}{'-log p':>8}{'flag':>7}"
print(f"{'':>18} | {'HIDDEN CELLS':^42} | {'VISIBLE CELLS':^42}")
print(f"{'col':>6}{'cells':>7}{'arm':>5} | {hdr} | {hdr}")
for col in COLOURS:
    for T in [6, 12, 18]:
        for arm in CAL_ARMS:
            h = [agg(arm,col,T,'hidden',x) for x in ('bit_acc','conf','conf_wrong','nll','flag')]
            v = [agg(arm,col,T,'visible',x) for x in ('bit_acc','conf','conf_wrong','nll','flag')]
            fmt = lambda q: (f'{100*q[0]:>11.2f}{q[1]:>7.3f}{100*q[2]:>8.1f}%{q[3]:>8.2f}' +
                             (f'{100*q[4]:>6.1f}%' if not np.isnan(q[4]) else f'{"":>7}'))
            print(f'{col:>6}{EV[(col,T)]["occ"].mean():>7.1f}{arm:>5} | {fmt(h)} | {fmt(v)}')
        print()


                   |                HIDDEN CELLS                |               VISIBLE CELLS               
   col  cells  arm |         acc   conf wrong>.9  -log p   flag |         acc   conf wrong>.9  -log p   flag
  grey    5.7    P |       50.15  0.998    49.6%    6.80        |       88.65  0.998    11.1%    1.47       
  grey    5.7    B |       57.46  0.684     3.0%    0.72        |       94.86  0.944     1.6%    0.16       
  grey    5.7    C |       52.99  0.964    42.1%    3.51  96.6% |       94.96  0.944     1.6%    0.16   1.3%
  grey    5.7    E |       57.28  0.664     1.6%    0.69        |       95.24  0.928     1.0%    0.15       
  grey    5.7    F |       52.46  0.977    44.2%    4.50  98.4% |       95.15  0.944     1.4%    0.15   6.1%

  grey   11.5    P |       53.95  0.999    45.8%    6.27        |       86.55  0.998    13.1%    1.75       
  grey   11.5    B |       55.47  0.634     2.0%    0.72        |       93.85  0.944     2.1%    0.18       
  grey   11.5    C

In [19]:
res = dict(code=CODE_TAG, seeds=SEEDS, margin=MARGIN,
           arms={k: {kk: str(vv) for kk,vv in v.items()} for k,v in ARMS.items()},
           levels={str(k): v for k,v in LEVEL_FOR.items()},
           seconds={f'{a}|{s}': float(t) for (a,s),t in TIMES.items()},
           accuracy_soft={f'{a}|{s}|{c}|{T}': float(ACC[(a,s,c,T)].mean()) for (a,s,c,T) in ACC},
           compare=CMP, cap_choice=CAP_CHOICE,
           cap={arm: {f'{s}|{e}|{k[0]}|{k[1]}|{sp}': v for (s,e,k,sp), v in out.items()} for arm, out in CAP.items()},
           damage=DAMAGE, clean_reference=clean_ref, calibration=CAL_ROWS)
with open(f'{ART}/n14_dev.json','w') as fh: json.dump(res, fh, indent=1)
print('saved', f'{ART}/n14_dev.json')


saved /content/drive/MyDrive/aruco_p2/artifacts/n14_dev.json
